# Module 4: AI Agents & Autonomous Tool Selection

Is notebook me hum AI Agent architecture ko detail me samjhenge:
1. **Intent Detection**: Student ke natural prompt ko identify karna (Answer, Summary, ya Quiz).
2. **Tools Architecture**:
   - Answer Tool: RAG-based grounded question answering.
   - Summary Tool: Study notes ka executive summary aur key bullet points.
   - Quiz Tool: Automated MCQs with explanations aur scoring.
3. **Autonomous Agent Orchestrator**: Sequential state machine jo user ke prompt ko evaluate karta hai, right tool execute karta hai, aur execution trace generate karta hai.

In [ ]:
import sys
from pathlib import Path

# Setup project root
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.agents.intent_detector import IntentDetector
from src.rag.pipeline import RAGPipeline
from src.agents.tools.answer_tool import AnswerTool
from src.agents.tools.summary_tool import SummaryTool
from src.agents.tools.quiz_tool import QuizTool
from src.agents.orchestrator import AgentOrchestrator

print("[OK] Successfully imported Module 4 agent components.")

## 1. Intent Detection Demonstration

IntentDetector prompt ke semantic aur structural signals ko observe karke student ki intent class aur relevant parameters (num_questions, topic, document) extract karta hai.

In [ ]:
detector = IntentDetector()

test_prompts = [
    "Explain Python lists and generators",
    "Summarize my ML notes from sample_study_notes.txt",
    "Generate 4 MCQs on machine learning regularization",
    "What is the role of Global Interpreter Lock in CPython?"
]

for prompt in test_prompts:
    res = detector.detect_intent(prompt)
    print(f"Prompt: '{prompt}'")
    print(f"  Intent:     {res['intent']} (Confidence: {res['confidence']})")
    print(f"  Reasoning:  {res['reasoning']}")
    print(f"  Parameters: {res['parameters']}\n")

## 2. Individual Tools Execution

Har ek tool standard input accept karta hai aur structured dictionary return karta hai.

In [ ]:
# Initialize RAG Pipeline and FAISS store
idx_path = str(project_root / "models" / "faiss_index.bin")
meta_path = str(project_root / "models" / "faiss_chunks.json")
rag = RAGPipeline(index_path=idx_path, metadata_path=meta_path, confidence_threshold=0.35)

# Tool 1: Summary Tool
summary_tool = SummaryTool(vector_store=rag.vector_store)
summary_res = summary_tool.execute(max_points=3)
print("[Summary Tool Result]")
print(f"Executive Summary: {summary_res['executive_summary']}")
print("Takeaways:")
for pt in summary_res['key_takeaways']:
    print(f"  - {pt}")

# Tool 2: Quiz Tool
quiz_tool = QuizTool(vector_store=rag.vector_store)
quiz_res = quiz_tool.execute(topic="Python", num_questions=2)
print("\n[Quiz Tool Result]")
for q in quiz_res['questions']:
    print(f"Q{q['id']}: {q['question']}")
    for opt in q['options']:
        print(f"   {opt}")
    print(f"   Correct: Option {q['correct_option']}\n")

## 3. Autonomous Agent Orchestrator

Orchestrator bina kisi manual button click ke student ke prompt ko analyze karke automatically right tool choose karta hai aur execution trace maintain karta hai.

In [ ]:
orchestrator = AgentOrchestrator(rag_pipeline=rag, vector_store=rag.vector_store)

response = orchestrator.run("Create 2 MCQs from Python notes")
print(f"[Intent Detected]: {response['intent']}")
print(f"[Tool Used]:       {response['tool_used']}")
print("\n[Execution Trace]:")
for step in response['execution_trace']:
    print(f"  {step}")

print("\n[Agent Final Output]:\n")
print(response['agent_response'])